## base_model--> non_instruction_model--> instruction_model--> preference_model

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [2]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [4]:
import zipfile
import os
# Path to your zip file
zip_path = "/content/tinyllama-instruction.zip"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [5]:
model_path = "/content/checkpoint-3"

In [6]:
!pip install -U torchao

In [7]:
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [8]:
prompt = "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."

In [9]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [10]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [11]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry.
2018-2023: PhD Student, Research Group of Dr. Thomas Sander, University of Oxford, United Kingdom. 2017-2018: Postdoctoral Associate, Dr. Thomas Sander, University of Oxford, United Kingdom. 2016-2017: Postdoctoral Associate, Dr. Manoj Kumar, National Institute of Dental and Craniofacial Research


## Now lets start with prefrence base tuning or preference based alignment

In [12]:
!pip install -U trl

In [13]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.0 MB/s eta 0:00:00


In [14]:
from trl import DPOTrainer
from transformers import AutoTokenizer,  AutoModelForCausalLM, TrainingArguments
from peft import PeftModel
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

In [15]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [27]:
instruction_checkpoint = "/content/checkpoint-3"

In [16]:
# Load dataset
dataset = load_dataset("csv", data_files="/content/pharma_preference_data.csv")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [17]:
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [18]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

get_peft_model() → Create a new LoRA during training

PeftModel.from_pretrained() → Load an already-trained LoRA for inference or further training

In [19]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [ ]:
# pref_model_lora = get_peft_model(instruction_model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [20]:
base_model

'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T'

In [24]:
# 8bit量化配置
from transformers import LlamaForCausalLM, BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_skip_modules=None
)

In [30]:
#STEP A: Load base
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [31]:
#STEP B: Load Instruction LoRA + merge
model = PeftModel.from_pretrained(model, instruction_checkpoint)

In [32]:
model = model.merge_and_unload()

In [33]:
#STEP C: Attach NEW LoRA for preference
pref_model_lora = get_peft_model(model, lora_config)

| Stage           | What You Should Do                       | Wrong Way (you did)      |
| --------------- | ---------------------------------------- | ------------------------ |
| Non-Instruction | Base + LoRA                              | ✔ correct                |
| Instruction     | Base + **merge(stage1 LoRA)** + NEW LoRA | ❌ “LoRA on LoRA”         |
| Preference      | Base + **merge(stage2 LoRA)** + NEW LoRA | ❌ “LoRA on LoRA on LoRA” |


In [34]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [35]:
from trl import DPOTrainer, DPOConfig

In [39]:
dpo_args = DPOConfig(
    output_dir="./tinyllama-preference-alignment",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    beta=0.1,
    report_to=[],
    logging_dir=None, # disable logging to wandb or tensorboard
    loss_type="sigmoid",  # or "hinge", depending on experiment
    remove_unused_columns=False
)


In [40]:
trainer = DPOTrainer(
    model=pref_model_lora,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    processing_class=tokenizer,   # instead of tokenizer argument
    # you can pass data_collator if needed,
    # optionally eval_dataset etc.
)

Dropping fully truncated examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

In [41]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss


TrainOutput(global_step=1, training_loss=0.6931471824645996, metrics={'train_runtime': 4.6022, 'train_samples_per_second': 1.086, 'train_steps_per_second': 0.217, 'total_flos': 4366854955008.0, 'train_loss': 0.6931471824645996, 'entropy': 2.0755802392959595, 'num_tokens': 566.0, 'logits/chosen': -3.5722855347523557, 'logits/rejected': -3.667063484447058, 'mean_token_accuracy': 0.5977408826351166, 'rewards/chosen': 0.0, 'rewards/rejected': 0.0, 'rewards/accuracies': 0.0, 'rewards/margins': 0.0, 'logps/chosen': -93.52379608154297, 'logps/rejected': -62.94301147460938, 'epoch': 1.0})

### Testing with Non-Instruction Model

In [42]:
question = "Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment."

In [45]:
import zipfile
import os
# Path to your zip file
zip_path = "/content/tinyllama-non-instruction.zip"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [46]:
model_path = "/content/checkpoint-5"
non_instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [47]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [48]:
outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [49]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
Crohn's disease (CD) is an inflammatory bowel disease that can be severe. Recent research has shown that metformin may reduce the risk of flare-ups in people with CD.
What Is Crohn's Disease?
Crohn's disease (CD) is a type of inflammatory bowel disease that is characterized by inflammation in the intestinal tract, which can lead to abdom


### Testing with Instruction-Fine-Tuned Model

In [54]:
model_path = "/content/checkpoint-3"
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="cuda")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [55]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [56]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [57]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
Metformin, a common diabetes drug used to treat type 2 diabetes, has long been known as a "fat burner," but new research suggests that it may also help prevent cancer.
In a review of more than 100 studies published this week in the journal Diabetologia, researchers from Britain's National Health Service (NHS) and Oxford University said metformin may have properties that can be exploited to prevent cancer by:


In [69]:
import torch
import gc

# 删除所有旧模型、训练器对象
del instruction_model,non_instruction_model
gc.collect()
torch.cuda.empty_cache()

### Testing with DPO (Preference-Aligned) Model

In [62]:
model_path = "/content/tinyllama-preference-alignment/checkpoint-1"

In [63]:
preference_aligned_model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float16)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [70]:
preference_aligned_model.to("cuda")

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=256, bia

In [71]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [72]:
outputs = preference_aligned_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [73]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
Medicines that are used to treat people with cancer, including chemotherapy drugs, hormone therapy medications, targeted therapies, and immunotherapy drugs. Some types of chemotherapy can also cause side effects such as fatigue or a cough. These side effects may be related to other parts of the body.
A substance that is naturally occurring or synthetically made in laboratories to produce new chemicals. Chem
